In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import ipywidgets as widgets
from IPython.display import display

plt.rcParams.update({
    'figure.facecolor': '#0f0f1a',
    'axes.facecolor': '#1a1a2e',
    'axes.edgecolor': '#444466',
    'axes.labelcolor': '#ccccee',
    'axes.grid': True,
    'grid.color': '#2a2a4a',
    'grid.linestyle': '--',
    'grid.alpha': 0.6,
    'xtick.color': '#888899',
    'ytick.color': '#888899',
    'text.color': '#ccccee',
    'legend.facecolor': '#1a1a2e',
    'legend.edgecolor': '#444466',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titlepad': 12,
})

COLORS = {
    'plasma':    '#7eb8f7',
    'tissue':    '#f97b6b',
    'linear':    '#7eb8f7',
    'nonlinear': '#f7c97e',
    'dose':      '#a8f5a2',
    'threshold': '#f97b6b',
}

print('Ready.')

In [ ]:
dose   = 500       # mg
Vd     = 50        # L
ke     = 0.15      # 1/hr
C0     = dose / Vd
t_half = np.log(2) / ke
t      = np.linspace(0, 48, 500)
C      = C0 * np.exp(-ke * t)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('Model 1 — One-Compartment IV', fontsize=15, fontweight='bold', color='#e0e0ff')

ax1.plot(t, C, color=COLORS['plasma'], lw=2.5, label='Plasma concentration')
ax1.axhline(C0/2, color=COLORS['threshold'], lw=1.2, ls='--', alpha=0.7, label=f'Half-max ({C0/2:.1f} mg/L)')
ax1.axvline(t_half, color=COLORS['threshold'], lw=1.2, ls=':', alpha=0.7, label=f't½ = {t_half:.1f} hr')
ax1.fill_between(t, C, alpha=0.08, color=COLORS['plasma'])
ax1.set_xlabel('Time (hr)')
ax1.set_ylabel('Concentration (mg/L)')
ax1.set_title('Linear Scale')
ax1.legend()

ax2.semilogy(t, C, color=COLORS['plasma'], lw=2.5)
ax2.axvline(t_half, color=COLORS['threshold'], lw=1.2, ls=':', alpha=0.7, label=f't½ = {t_half:.1f} hr')
ax2.set_xlabel('Time (hr)')
ax2.set_ylabel('Concentration (mg/L) — log scale')
ax2.set_title('Log Scale (straight line = first-order kinetics)')
ax2.legend()

fig.tight_layout()
plt.show()

print(f'C₀: {C0:.1f} mg/L  |  ke: {ke} hr⁻¹  |  t½: {t_half:.1f} hr')

In [ ]:
dose = 500
Vd   = 50
ke   = 0.15
ka   = 0.80
F    = 0.75

def oral_pk(t, y, ka, ke, Vd, F, dose):
    A_gut, C = y
    dA_gut = -ka * A_gut
    dC     = (F * ka * A_gut) / Vd - ke * C
    return [dA_gut, dC]

t_span = (0, 48)
t_eval = np.linspace(0, 48, 1000)
y0     = [dose, 0]

sol      = solve_ivp(oral_pk, t_span, y0, t_eval=t_eval, args=(ka, ke, Vd, F, dose), rtol=1e-8)
C_plasma = sol.y[1]
A_gut    = sol.y[0]
Cmax_idx = np.argmax(C_plasma)
Cmax     = C_plasma[Cmax_idx]
Tmax     = sol.t[Cmax_idx]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('Model 2 — One-Compartment Oral', fontsize=15, fontweight='bold', color='#e0e0ff')

ax1.plot(sol.t, C_plasma, color=COLORS['plasma'], lw=2.5, label='Plasma concentration')
ax1.plot(sol.t, A_gut / Vd, color=COLORS['tissue'], lw=1.8, ls='--', alpha=0.7, label='Gut amount (scaled)')
ax1.axvline(Tmax, color=COLORS['threshold'], lw=1.2, ls=':', label=f'Tmax = {Tmax:.1f} hr')
ax1.axhline(Cmax, color=COLORS['threshold'], lw=1.2, ls='--', alpha=0.6, label=f'Cmax = {Cmax:.1f} mg/L')
ax1.fill_between(sol.t, C_plasma, alpha=0.08, color=COLORS['plasma'])
ax1.set_xlabel('Time (hr)')
ax1.set_ylabel('Concentration (mg/L)')
ax1.set_title('Plasma Concentration vs Time')
ax1.legend()

for f_val, alpha in [(1.0, 1.0), (0.75, 0.8), (0.5, 0.6), (0.25, 0.4)]:
    sol_f = solve_ivp(oral_pk, t_span, [dose, 0], t_eval=t_eval, args=(ka, ke, Vd, f_val, dose), rtol=1e-8)
    ax2.plot(sol_f.t, sol_f.y[1], color=COLORS['plasma'], alpha=alpha, lw=2, label=f'F = {f_val:.0%}')

ax2.set_xlabel('Time (hr)')
ax2.set_ylabel('Concentration (mg/L)')
ax2.set_title('Effect of Bioavailability (F)')
ax2.legend()

fig.tight_layout()
plt.show()

print(f'Cmax: {Cmax:.2f} mg/L  |  Tmax: {Tmax:.2f} hr  |  Bioavailability: {F:.0%}')

In [ ]:
dose = 500
V1   = 10
V2   = 40
k12  = 0.30
k21  = 0.10
ke   = 0.12

def two_compartment(t, y, V1, V2, k12, k21, ke):
    C1, C2 = y
    dC1 = -(k12 + ke) * C1 + k21 * C2 * (V2 / V1)
    dC2 = k12 * C1 * (V1 / V2) - k21 * C2
    return [dC1, dC2]

t_span = (0, 72)
t_eval = np.linspace(0, 72, 2000)
C1_0   = dose / V1
sol    = solve_ivp(two_compartment, t_span, [C1_0, 0], t_eval=t_eval,
                   args=(V1, V2, k12, k21, ke), rtol=1e-9)
C1 = sol.y[0]
C2 = sol.y[1]
C_1comp = (dose / (V1 + V2)) * np.exp(-ke * t_eval)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('Model 3 — Two-Compartment IV', fontsize=15, fontweight='bold', color='#e0e0ff')

ax1.plot(sol.t, C1, color=COLORS['plasma'], lw=2.5, label='Central (plasma)')
ax1.plot(sol.t, C2, color=COLORS['tissue'], lw=2.5, label='Peripheral (tissue)')
ax1.fill_between(sol.t, C1, alpha=0.07, color=COLORS['plasma'])
ax1.fill_between(sol.t, C2, alpha=0.07, color=COLORS['tissue'])
ax1.set_xlabel('Time (hr)')
ax1.set_ylabel('Concentration (mg/L)')
ax1.set_title('Central vs Peripheral Compartment')
ax1.legend()

ax2.semilogy(sol.t, C1, color=COLORS['plasma'], lw=2.5, label='2-compartment (central)')
ax2.semilogy(t_eval, C_1comp, color='#888899', lw=1.8, ls='--', alpha=0.8, label='1-compartment reference')
ax2.annotate('Distribution\nphase (α)', xy=(4, C1[80]), xytext=(8, C1[80]*3),
             color='#aaaacc', fontsize=9, arrowprops=dict(arrowstyle='->', color='#666688'))
ax2.annotate('Elimination\nphase (β)', xy=(40, C1[1100]), xytext=(50, C1[1100]*4),
             color='#aaaacc', fontsize=9, arrowprops=dict(arrowstyle='->', color='#666688'))
ax2.set_xlabel('Time (hr)')
ax2.set_ylabel('Concentration (mg/L) — log scale')
ax2.set_title('Log Scale — Biphasic Decay')
ax2.legend()

fig.tight_layout()
plt.show()

In [ ]:
Vd   = 50
ke   = 0.15
Vmax = 8.0
Km   = 5.0

def mm_pk(t, y, Vmax, Km):
    C = y[0]
    dC = -(Vmax * C) / (Km + C)
    return [dC]

t_span  = (0, 48)
t_eval  = np.linspace(0, 48, 2000)
doses   = [50, 200, 500]
d_colors = ['#a8f5a2', '#7eb8f7', '#f7c97e']

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
fig.suptitle('Model 4 — Michaelis-Menten vs Linear Elimination', fontsize=15, fontweight='bold', color='#e0e0ff')

for ax, dose, color in zip(axes, doses, d_colors):
    C0 = dose / Vd
    C_linear = C0 * np.exp(-ke * t_eval)
    sol_mm = solve_ivp(mm_pk, t_span, [C0], t_eval=t_eval, args=(Vmax, Km), rtol=1e-9)

    ax.semilogy(t_eval, C_linear, color=COLORS['linear'], lw=2, ls='--', label='Linear')
    ax.semilogy(sol_mm.t, sol_mm.y[0], color=color, lw=2.5, label='Michaelis-Menten')
    ax.axhline(Km, color='#ff8888', lw=1, ls=':', alpha=0.7, label=f'Km = {Km} mg/L')

    sat = (C0 / (C0 + Km)) * 100
    ax.set_title(f'Dose = {dose} mg  (C₀ = {C0:.0f} mg/L)\nInitial enzyme saturation: {sat:.0f}%')
    ax.set_xlabel('Time (hr)')
    if ax is axes[0]:
        ax.set_ylabel('Concentration (mg/L)')
    ax.legend(fontsize=9)

fig.tight_layout()
plt.show()

In [ ]:
dose  = 200
Vd    = 50
ke    = 0.12
ka    = 0.70
F     = 0.80
tau   = 8
n     = 10
t_half = np.log(2) / ke

def oral_ode(t, y):
    A, C = y
    return [-ka * A, (F * ka * A) / Vd - ke * C]

def simulate_multidose(dose, tau, n_doses):
    t_all, C_all = [], []
    y_cur = [0, 0]
    for i in range(n_doses):
        t_s = i * tau
        t_e = t_s + tau
        y_cur[0] += dose
        sol = solve_ivp(oral_ode, (t_s, t_e), y_cur,
                        t_eval=np.linspace(t_s, t_e, 300), rtol=1e-9)
        t_all.extend(sol.t)
        C_all.extend(sol.y[1])
        y_cur = [sol.y[0][-1], sol.y[1][-1]]
    return np.array(t_all), np.array(C_all)

t_all, C_all = simulate_multidose(dose, tau, n)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('Model 5 — Multiple Dosing & Accumulation', fontsize=15, fontweight='bold', color='#e0e0ff')

ax1.plot(t_all, C_all, color=COLORS['plasma'], lw=2.2)
ax1.fill_between(t_all, C_all, alpha=0.08, color=COLORS['plasma'])
for i in range(n):
    ax1.axvline(i * tau, color=COLORS['dose'], lw=0.8, alpha=0.5)
ax1.set_xlabel('Time (hr)')
ax1.set_ylabel('Plasma Concentration (mg/L)')
ax1.set_title(f'Every {tau} hr dosing — {n} doses')

for tau_val, alpha in [(4, 1.0), (8, 0.75), (12, 0.5), (24, 0.3)]:
    t_s, C_s = simulate_multidose(dose, tau_val, 12)
    ax2.plot(t_s, C_s, color=COLORS['plasma'], alpha=alpha, lw=1.8, label=f'Every {tau_val} hr')

ax2.set_xlabel('Time (hr)')
ax2.set_ylabel('Plasma Concentration (mg/L)')
ax2.set_title('Effect of Dosing Interval')
ax2.legend()

fig.tight_layout()
plt.show()

print(f't½ = {t_half:.1f} hr  |  Steady state after ~{4*t_half:.0f}–{5*t_half:.0f} hr')

In [ ]:
def plot_interactive(dose, Vd, ka, ke, F):
    t_eval = np.linspace(0, 72, 1000)

    def ode(t, y):
        A, C = y
        return [-ka * A, (F * ka * A) / Vd - ke * C]

    sol  = solve_ivp(ode, (0, 72), [dose, 0], t_eval=t_eval, rtol=1e-9)
    C    = sol.y[1]
    t_h  = np.log(2) / ke
    ci   = np.argmax(C)

    fig, ax = plt.subplots(figsize=(10, 4))
    fig.patch.set_facecolor('#0f0f1a')
    ax.set_facecolor('#1a1a2e')
    ax.plot(sol.t, C, color=COLORS['plasma'], lw=2.5)
    ax.fill_between(sol.t, C, alpha=0.1, color=COLORS['plasma'])
    ax.axvline(sol.t[ci], color=COLORS['threshold'], lw=1.2, ls=':', label=f'Tmax = {sol.t[ci]:.1f} hr')
    ax.axhline(C[ci], color=COLORS['threshold'], lw=1.2, ls='--', alpha=0.6, label=f'Cmax = {C[ci]:.2f} mg/L')
    ax.set_xlabel('Time (hr)', color='#ccccee')
    ax.set_ylabel('Plasma Concentration (mg/L)', color='#ccccee')
    ax.set_title(f't½ = {t_h:.1f} hr  |  AUC ≈ {np.trapz(C, sol.t):.0f} mg·hr/L', color='#e0e0ff')
    ax.tick_params(colors='#888899')
    ax.grid(True, color='#2a2a4a', ls='--', alpha=0.6)
    ax.spines[:].set_color('#444466')
    ax.legend()
    plt.tight_layout()
    plt.show()

style  = {'description_width': '150px'}
layout = widgets.Layout(width='500px')

w = dict(
    dose = widgets.FloatSlider(value=500,  min=50,   max=1500, step=50,   description='Dose (mg)',        style=style, layout=layout),
    Vd   = widgets.FloatSlider(value=50,   min=5,    max=200,  step=5,    description='Vd (L)',           style=style, layout=layout),
    ka   = widgets.FloatSlider(value=0.8,  min=0.1,  max=3.0,  step=0.1,  description='ka (1/hr)',        style=style, layout=layout),
    ke   = widgets.FloatSlider(value=0.15, min=0.01, max=0.8,  step=0.01, description='ke (1/hr)',        style=style, layout=layout),
    F    = widgets.FloatSlider(value=0.75, min=0.1,  max=1.0,  step=0.05, description='Bioavailability F',style=style, layout=layout),
)

out = widgets.interactive_output(plot_interactive, w)
display(widgets.VBox([*w.values(), out]))

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
dose  = 200    # mg
Vd    = 50     # L
ke    = 0.12   # 1/hr
ka    = 0.70   # 1/hr
F     = 0.80
tau   = 8      # hr — dosing interval
n     = 10     # number of doses

MEC   = 2.0    # mg/L — minimum effective concentration
MTC   = 7.0    # mg/L — maximum tolerated concentration

def oral_ode(t, y):
    A, C = y
    return [-ka * A, (F * ka * A) / Vd - ke * C]

def simulate_multidose(dose, tau, n_doses):
    t_all, C_all = [], []
    y_cur = [0, 0]
    for i in range(n_doses):
        t_s = i * tau
        t_e = t_s + tau
        y_cur[0] += dose
        sol = solve_ivp(oral_ode, (t_s, t_e), y_cur,
                        t_eval=np.linspace(t_s, t_e, 300), rtol=1e-9)
        t_all.extend(sol.t)
        C_all.extend(sol.y[1])
        y_cur = [sol.y[0][-1], sol.y[1][-1]]
    return np.array(t_all), np.array(C_all)

# ── Plot three dosing regimens side by side ────────────────────────────────────
regimens = [
    {'tau': 4,  'dose': 100, 'label': 'Every 4 hr / 100 mg'},
    {'tau': 8,  'dose': 200, 'label': 'Every 8 hr / 200 mg'},
    {'tau': 24, 'dose': 500, 'label': 'Every 24 hr / 500 mg'},
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Therapeutic Window — Dosing Regimen Comparison',
             fontsize=15, fontweight='bold', color='#e0e0ff')

for ax, reg in zip(axes, regimens):
    t_out, C_out = simulate_multidose(reg['dose'], reg['tau'], n)

    # Shade zones
    ax.axhspan(0,    MEC, alpha=0.10, color='#f97b6b', label='Sub-therapeutic')
    ax.axhspan(MEC,  MTC, alpha=0.10, color='#a8f5a2', label='Therapeutic window')
    ax.axhspan(MTC,  max(C_out) * 1.3 + 1, alpha=0.10, color='#f7c97e', label='Toxic zone')

    # Boundary lines
    ax.axhline(MEC, color='#f97b6b', lw=1.3, ls='--', alpha=0.8)
    ax.axhline(MTC, color='#f7c97e', lw=1.3, ls='--', alpha=0.8)

    # Concentration curve
    ax.plot(t_out, C_out, color=COLORS['plasma'], lw=2.3)
    ax.fill_between(t_out, C_out, alpha=0.08, color=COLORS['plasma'])

    # Dose tick marks
    for i in range(n):
        ax.axvline(i * reg['tau'], color=COLORS['dose'], lw=0.7, alpha=0.4)

    # Annotate MEC and MTC
    ax.text(t_out[-1] * 0.98, MEC + 0.15, 'MEC', color='#f97b6b',
            fontsize=8, ha='right', va='bottom')
    ax.text(t_out[-1] * 0.98, MTC + 0.15, 'MTC', color='#f7c97e',
            fontsize=8, ha='right', va='bottom')

    # Work out how much time is spent in each zone
    in_window = np.sum((C_out >= MEC) & (C_out <= MTC)) / len(C_out) * 100
    toxic     = np.sum(C_out > MTC) / len(C_out) * 100
    sub       = np.sum(C_out < MEC) / len(C_out) * 100

    ax.set_title(f"{reg['label']}\n"
                 f"In window: {in_window:.0f}%  |  Toxic: {toxic:.0f}%  |  Sub: {sub:.0f}%",
                 fontsize=10)
    ax.set_xlabel('Time (hr)')
    if ax is axes[0]:
        ax.set_ylabel('Plasma Concentration (mg/L)')
    ax.set_ylim(bottom=0)
    ax.legend(fontsize=8, loc='upper left')

fig.tight_layout()
plt.show()

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

# ── Base parameters ────────────────────────────────────────────────────────────
n_patients  = 200
dose        = 300     # mg
tau         = 12      # hr dosing interval
n_doses     = 6
t_eval      = np.linspace(0, tau * n_doses, 800)

# Realistic population distributions (mean, coefficient of variation)
np.random.seed(42)
ka_pop  = np.random.lognormal(mean=np.log(0.8),  sigma=0.3,  size=n_patients)
ke_pop  = np.random.lognormal(mean=np.log(0.12), sigma=0.25, size=n_patients)
Vd_pop  = np.random.lognormal(mean=np.log(50),   sigma=0.3,  size=n_patients)
F_pop   = np.clip(np.random.normal(0.75, 0.10,   n_patients), 0.2, 1.0)

MEC = 2.0
MTC = 7.0

def simulate_patient(ka, ke, Vd, F, dose, tau, n_doses):
    def ode(t, y):
        A, C = y
        return [-ka * A, (F * ka * A) / Vd - ke * C]
    t_all, C_all = [], []
    y_cur = [0, 0]
    for i in range(n_doses):
        t_s = i * tau
        t_e = t_s + tau
        y_cur[0] += dose
        sol = solve_ivp(ode, (t_s, t_e), y_cur,
                        t_eval=np.linspace(t_s, t_e, 150), rtol=1e-8)
        t_all.extend(sol.t)
        C_all.extend(sol.y[1])
        y_cur = [sol.y[0][-1], sol.y[1][-1]]
    return np.array(t_all), np.array(C_all)

# ── Simulate all patients ──────────────────────────────────────────────────────
all_curves = []
for i in range(n_patients):
    t_p, C_p = simulate_patient(ka_pop[i], ke_pop[i], Vd_pop[i], F_pop[i],
                                 dose, tau, n_doses)
    all_curves.append((t_p, C_p))

# Common time axis for statistics
t_common = all_curves[0][0]
C_matrix = np.array([np.interp(t_common, t_p, C_p) for t_p, C_p in all_curves])

C_mean   = np.mean(C_matrix, axis=0)
C_p5     = np.percentile(C_matrix, 5,  axis=0)
C_p25    = np.percentile(C_matrix, 25, axis=0)
C_p75    = np.percentile(C_matrix, 75, axis=0)
C_p95    = np.percentile(C_matrix, 95, axis=0)

# Per-patient stats
Cmax_all = C_matrix.max(axis=1)
pct_toxic = np.mean(C_matrix.max(axis=1) > MTC) * 100
pct_sub   = np.mean(C_matrix.min(axis=1) < MEC) * 100

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle(f'Population Variability — {n_patients} Simulated Patients',
             fontsize=15, fontweight='bold', color='#e0e0ff')

# Left — individual curves + population statistics
ax = axes[0]
for t_p, C_p in all_curves:
    ax.plot(t_p, C_p, color=COLORS['plasma'], alpha=0.04, lw=0.8)

ax.fill_between(t_common, C_p5,  C_p95, alpha=0.15, color=COLORS['plasma'], label='5th–95th percentile')
ax.fill_between(t_common, C_p25, C_p75, alpha=0.25, color=COLORS['plasma'], label='25th–75th percentile')
ax.plot(t_common, C_mean, color='white', lw=2.5, label='Population mean')

ax.axhline(MEC, color='#f97b6b', lw=1.3, ls='--', alpha=0.8, label=f'MEC = {MEC} mg/L')
ax.axhline(MTC, color='#f7c97e', lw=1.3, ls='--', alpha=0.8, label=f'MTC = {MTC} mg/L')
ax.axhspan(MEC, MTC, alpha=0.05, color='#a8f5a2')

for i in range(n_doses):
    ax.axvline(i * tau, color=COLORS['dose'], lw=0.7, alpha=0.4)

ax.set_xlabel('Time (hr)')
ax.set_ylabel('Plasma Concentration (mg/L)')
ax.set_title(f'Dose: {dose} mg every {tau} hr\n'
             f'Toxic peak: {pct_toxic:.0f}% of patients  |  Sub-therapeutic trough: {pct_sub:.0f}%')
ax.set_ylim(bottom=0)
ax.legend(fontsize=9)

# Right — Cmax distribution histogram
ax2 = axes[1]
counts, bins, patches = ax2.hist(Cmax_all, bins=30, color=COLORS['plasma'],
                                  alpha=0.7, edgecolor='#1a1a2e', linewidth=0.5)

# Color bars by zone
for patch, left in zip(patches, bins[:-1]):
    if left > MTC:
        patch.set_facecolor('#f7c97e')
        patch.set_alpha(0.85)
    elif left < MEC:
        patch.set_facecolor('#f97b6b')
        patch.set_alpha(0.85)

ax2.axvline(MEC, color='#f97b6b', lw=1.5, ls='--', label=f'MEC = {MEC} mg/L')
ax2.axvline(MTC, color='#f7c97e', lw=1.5, ls='--', label=f'MTC = {MTC} mg/L')
ax2.axvline(np.median(Cmax_all), color='white', lw=1.8, ls=':',
            label=f'Median Cmax = {np.median(Cmax_all):.1f} mg/L')

ax2.set_xlabel('Peak Concentration Cmax (mg/L)')
ax2.set_ylabel('Number of Patients')
ax2.set_title('Distribution of Individual Peak Concentrations\n'
              f'Blue = therapeutic  |  Red = sub-therapeutic  |  Yellow = toxic')
ax2.legend(fontsize=9)

fig.tight_layout()
plt.show()

print(f'Population mean Cmax: {np.mean(Cmax_all):.2f} mg/L')
print(f'Patients exceeding MTC (toxic):       {pct_toxic:.1f}%')
print(f'Patients below MEC (sub-therapeutic): {pct_sub:.1f}%')

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# ── Drug database ──────────────────────────────────────────────────────────────
# All values are real-world population mean PK parameters from literature
DRUGS = {
    'Ibuprofen': {
        'ka': 1.5,    'ke': 0.45,   'Vd': 10,   'F': 0.80,
        'Km': None,   'Vmax': None,
        'dose': 400,  'tau': 6,     'MEC': 10.0, 'MTC': 50.0,
        'unit': 'mg/L',
        'notes': 'NSAID analgesic. Rapid absorption, short half-life (~1.5 hr). '
                 'MEC for analgesia ~10 mg/L. Gastric irritation risk at high concentrations.',
        'compartments': 1, 'route': 'oral',
    },
    'Caffeine': {
        'ka': 0.90,   'ke': 0.10,   'Vd': 38,   'F': 1.00,
        'Km': None,   'Vmax': None,
        'dose': 200,  'tau': 6,     'MEC': 0.003, 'MTC': 0.08,
        'unit': 'mg/L',
        'notes': 'Near-complete oral bioavailability (F ≈ 1.0). Half-life ~5–6 hr. '
                 'Widely distributed. Metabolized by CYP1A2 in the liver.',
        'compartments': 1, 'route': 'oral',
    },
    'Amoxicillin': {
        'ka': 1.2,    'ke': 0.60,   'Vd': 20,   'F': 0.90,
        'Km': None,   'Vmax': None,
        'dose': 500,  'tau': 8,     'MEC': 0.5,  'MTC': 50.0,
        'unit': 'mg/L',
        'notes': 'Beta-lactam antibiotic. Good oral bioavailability. Short half-life (~1 hr). '
                 'Efficacy is time-dependent — concentration must stay above MEC throughout dosing interval.',
        'compartments': 1, 'route': 'oral',
    },
    'Warfarin': {
        'ka': 0.50,   'ke': 0.021,  'Vd': 8,    'F': 0.95,
        'Km': None,   'Vmax': None,
        'dose': 5,    'tau': 24,    'MEC': 0.5,  'MTC': 2.5,
        'unit': 'mg/L',
        'notes': 'Anticoagulant with narrow therapeutic window. Long half-life (~33 hr). '
                 'Takes ~5 days to reach steady state. Highly protein-bound. '
                 'Small dose changes have large clinical consequences.',
        'compartments': 1, 'route': 'oral',
    },
    'Morphine': {
        'ka': 0.80,   'ke': 0.25,   'Vd': 200,  'F': 0.30,
        'Km': None,   'Vmax': None,
        'dose': 10,   'tau': 4,     'MEC': 0.01, 'MTC': 0.07,
        'unit': 'mg/L',
        'notes': 'Opioid analgesic. Low oral bioavailability (~30%) due to extensive first-pass metabolism. '
                 'Large volume of distribution — distributes widely into tissues. '
                 'Narrow therapeutic window requires careful dosing.',
        'compartments': 1, 'route': 'oral',
    },
    'Metformin': {
        'ka': 0.30,   'ke': 0.14,   'Vd': 100,  'F': 0.55,
        'Km': None,   'Vmax': None,
        'dose': 1000, 'tau': 12,    'MEC': 0.5,  'MTC': 4.0,
        'unit': 'mg/L',
        'notes': 'Type 2 diabetes medication. Slow absorption (ka lower than most). '
                 'Moderate bioavailability. Not metabolized — excreted unchanged by kidneys. '
                 'Renal impairment significantly raises exposure and lactic acidosis risk.',
        'compartments': 1, 'route': 'oral',
    },
    'Phenytoin (Dilantin)': {
        'ka': 0.25,   'ke': None,   'Vd': 45,   'F': 0.90,
        'Km': 5.0,    'Vmax': 7.0,
        'dose': 300,  'tau': 24,    'MEC': 10.0, 'MTC': 20.0,
        'unit': 'mg/L',
        'notes': 'Antiepileptic. Classic example of Michaelis-Menten (nonlinear) elimination. '
                 'Enzymes saturate within the therapeutic range — small dose increases cause '
                 'disproportionately large concentration increases. Requires careful monitoring.',
        'compartments': 1, 'route': 'oral',
    },
    'Vancomycin': {
        'ka': None,   'ke': 0.13,   'Vd': 50,   'F': None,
        'Km': None,   'Vmax': None,
        'dose': 1000, 'tau': 12,    'MEC': 10.0, 'MTC': 40.0,
        'unit': 'mg/L',
        'notes': 'IV antibiotic — not orally bioavailable. Two-compartment behavior. '
                 'Eliminated almost entirely by the kidneys. Narrow window: '
                 'troughs must stay above MEC to prevent resistance, below MTC to prevent nephrotoxicity.',
        'compartments': 2, 'route': 'iv',
        'k12': 0.15, 'k21': 0.08, 'V1': 15, 'V2': 35,
    },
}

COLORS = {
    'plasma':    '#7eb8f7',
    'tissue':    '#f97b6b',
    'threshold': '#f97b6b',
    'dose':      '#a8f5a2',
    'window':    '#a8f5a2',
    'toxic':     '#f7c97e',
}

# ── Simulation helpers ─────────────────────────────────────────────────────────
def simulate_oral_multidose(ka, ke, Vd, F, dose, tau, n_doses, Km=None, Vmax=None):
    use_mm = (Km is not None and Vmax is not None)

    def ode(t, y):
        A, C = y
        if use_mm:
            elim = (Vmax * C) / (Km + C)
        else:
            elim = ke * C
        return [-ka * A, (F * ka * A) / Vd - elim]

    t_all, C_all = [], []
    y_cur = [0, 0]
    for i in range(n_doses):
        t_s = i * tau
        t_e = t_s + tau
        y_cur[0] += dose
        sol = solve_ivp(ode, (t_s, t_e), y_cur,
                        t_eval=np.linspace(t_s, t_e, 300), rtol=1e-9)
        t_all.extend(sol.t)
        C_all.extend(sol.y[1])
        y_cur = [sol.y[0][-1], sol.y[1][-1]]
    return np.array(t_all), np.array(C_all)

def simulate_iv_two_compartment(V1, V2, k12, k21, ke, dose, tau, n_doses):
    def ode(t, y):
        C1, C2 = y
        dC1 = -(k12 + ke) * C1 + k21 * C2 * (V2 / V1)
        dC2 = k12 * C1 * (V1 / V2) - k21 * C2
        return [dC1, dC2]

    t_all, C1_all, C2_all = [], [], []
    y_cur = [dose / V1, 0]
    for i in range(n_doses):
        t_s = i * tau
        t_e = t_s + tau
        sol = solve_ivp(ode, (t_s, t_e), y_cur,
                        t_eval=np.linspace(t_s, t_e, 300), rtol=1e-9)
        t_all.extend(sol.t)
        C1_all.extend(sol.y[0])
        C2_all.extend(sol.y[1])
        y_cur = [sol.y[0][-1], sol.y[1][-1]]
    return np.array(t_all), np.array(C1_all), np.array(C2_all)

def simulate_population(ka, ke, Vd, F, dose, tau, n_doses, n_patients=150,
                        Km=None, Vmax=None):
    np.random.seed(7)
    ka_pop  = np.random.lognormal(np.log(ka),  0.30, n_patients)
    ke_pop  = np.random.lognormal(np.log(ke),  0.25, n_patients) if ke else None
    Vd_pop  = np.random.lognormal(np.log(Vd),  0.30, n_patients)
    F_pop   = np.clip(np.random.normal(F, 0.08, n_patients), 0.1, 1.0)
    Vmax_pop = np.random.lognormal(np.log(Vmax), 0.20, n_patients) if Vmax else None

    curves = []
    for i in range(n_patients):
        ke_i   = ke_pop[i]  if ke_pop   is not None else ke
        Vmax_i = Vmax_pop[i] if Vmax_pop is not None else None
        t_p, C_p = simulate_oral_multidose(
            ka_pop[i], ke_i, Vd_pop[i], F_pop[i],
            dose, tau, n_doses, Km=Km, Vmax=Vmax_i
        )
        curves.append((t_p, C_p))

    t_ref = curves[0][0]
    C_mat = np.array([np.interp(t_ref, t_p, C_p) for t_p, C_p in curves])
    return t_ref, C_mat, curves

# ── Main dashboard function ────────────────────────────────────────────────────
def drug_dashboard(drug_name, n_doses, dose_override, show_population):
    p = DRUGS[drug_name]

    dose  = dose_override if dose_override > 0 else p['dose']
    tau   = p['tau']
    MEC   = p['MEC']
    MTC   = p['MTC']
    is_mm = (p['Km'] is not None)
    is_iv = (p['route'] == 'iv')
    is_2c = (p['compartments'] == 2)

    # ── Layout ────────────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(16, 10))
    fig.patch.set_facecolor('#0f0f1a')

    gs = fig.add_gridspec(2, 3, hspace=0.42, wspace=0.32,
                          left=0.06, right=0.97, top=0.88, bottom=0.07)

    ax_main  = fig.add_subplot(gs[0, :2])   # top left — main PK curve (wide)
    ax_log   = fig.add_subplot(gs[0, 2])    # top right — log scale
    ax_ss    = fig.add_subplot(gs[1, 0])    # bottom left — steady state zoom
    ax_pop   = fig.add_subplot(gs[1, 1])    # bottom middle — population
    ax_hist  = fig.add_subplot(gs[1, 2])    # bottom right — Cmax histogram

    for ax in [ax_main, ax_log, ax_ss, ax_pop, ax_hist]:
        ax.set_facecolor('#1a1a2e')
        ax.tick_params(colors='#888899')
        ax.spines[:].set_color('#444466')
        ax.xaxis.label.set_color('#ccccee')
        ax.yaxis.label.set_color('#ccccee')
        ax.title.set_color('#ddddff')
        ax.grid(True, color='#2a2a4a', ls='--', alpha=0.5)

    def shade_window(ax, y_top=None):
        if y_top is None:
            y_top = ax.get_ylim()[1]
        ax.axhspan(0,   MEC, alpha=0.08, color='#f97b6b')
        ax.axhspan(MEC, MTC, alpha=0.08, color='#a8f5a2')
        ax.axhspan(MTC, max(y_top, MTC * 1.5), alpha=0.08, color='#f7c97e')
        ax.axhline(MEC, color='#f97b6b', lw=1.1, ls='--', alpha=0.75)
        ax.axhline(MTC, color='#f7c97e', lw=1.1, ls='--', alpha=0.75)

    # ── Simulate ──────────────────────────────────────────────────────────────
    if is_2c:
        t_out, C1_out, C2_out = simulate_iv_two_compartment(
            p['V1'], p['V2'], p['k12'], p['k21'], p['ke'], dose, tau, n_doses)
        C_out = C1_out
    else:
        t_out, C_out = simulate_oral_multidose(
            p['ka'], p['ke'], p['Vd'], p['F'], dose, tau, n_doses,
            Km=p['Km'], Vmax=p['Vmax'])

    # ── Panel 1: Main PK curve ────────────────────────────────────────────────
    shade_window(ax_main, y_top=C_out.max() * 1.2)
    ax_main.plot(t_out, C_out, color=COLORS['plasma'], lw=2.5, label='Plasma concentration')
    if is_2c:
        ax_main.plot(t_out, C2_out, color=COLORS['tissue'], lw=1.8,
                     ls='--', alpha=0.7, label='Peripheral (tissue)')
    ax_main.fill_between(t_out, C_out, alpha=0.08, color=COLORS['plasma'])
    for i in range(n_doses):
        ax_main.axvline(i * tau, color=COLORS['dose'], lw=0.7, alpha=0.35)

    ax_main.set_xlabel('Time (hr)')
    ax_main.set_ylabel(f'Concentration ({p["unit"]})')

    route_label = 'IV' if is_iv else 'Oral'
    elim_label  = 'Michaelis-Menten' if is_mm else 'First-order'
    t_half_str  = f'{np.log(2)/p["ke"]:.1f} hr' if p['ke'] else 'nonlinear'
    ax_main.set_title(
        f'{drug_name} — {dose} mg {route_label} every {tau} hr × {n_doses} doses   '
        f'|   t½ ≈ {t_half_str}   |   Elimination: {elim_label}',
        fontsize=11
    )
    ax_main.text(0.01, 0.97, 'Toxic', transform=ax_main.transAxes,
                 color='#f7c97e', fontsize=8, va='top')
    ax_main.text(0.01, 0.60, 'Therapeutic', transform=ax_main.transAxes,
                 color='#a8f5a2', fontsize=8, va='top')
    ax_main.text(0.01, 0.15, 'Sub-therapeutic', transform=ax_main.transAxes,
                 color='#f97b6b', fontsize=8, va='top')
    ax_main.set_ylim(bottom=0)
    if is_2c:
        ax_main.legend(fontsize=9)

    # ── Panel 2: Log scale ────────────────────────────────────────────────────
    ax_log.semilogy(t_out, C_out, color=COLORS['plasma'], lw=2)
    ax_log.axhline(MEC, color='#f97b6b', lw=1.1, ls='--', alpha=0.75, label=f'MEC = {MEC}')
    ax_log.axhline(MTC, color='#f7c97e', lw=1.1, ls='--', alpha=0.75, label=f'MTC = {MTC}')
    ax_log.set_xlabel('Time (hr)')
    ax_log.set_ylabel(f'Concentration — log scale')
    ax_log.set_title('Log Scale')
    ax_log.legend(fontsize=8)

    # ── Panel 3: Steady-state zoom (last 2 doses) ─────────────────────────────
    cutoff = t_out[-1] - 2 * tau
    mask   = t_out >= cutoff
    shade_window(ax_ss, y_top=C_out[mask].max() * 1.3)
    ax_ss.plot(t_out[mask], C_out[mask], color=COLORS['plasma'], lw=2.2)
    ax_ss.fill_between(t_out[mask], C_out[mask], alpha=0.1, color=COLORS['plasma'])

    in_win = np.mean((C_out[mask] >= MEC) & (C_out[mask] <= MTC)) * 100
    toxic  = np.mean(C_out[mask] > MTC) * 100
    sub    = np.mean(C_out[mask] < MEC) * 100

    ax_ss.set_xlabel('Time (hr)')
    ax_ss.set_ylabel(f'Concentration ({p["unit"]})')
    ax_ss.set_title(f'Steady-State (last 2 doses)\n'
                    f'Window: {in_win:.0f}%  Toxic: {toxic:.0f}%  Sub: {sub:.0f}%',
                    fontsize=9)

    # ── Panels 4 & 5: Population (oral only) ──────────────────────────────────
    if show_population and not is_iv:
        t_ref, C_mat, curves = simulate_population(
            p['ka'], p['ke'] if p['ke'] else 0.12,
            p['Vd'], p['F'], dose, tau, n_doses,
            Km=p['Km'], Vmax=p['Vmax']
        )
        C_p5  = np.percentile(C_mat, 5,  axis=0)
        C_p25 = np.percentile(C_mat, 25, axis=0)
        C_p75 = np.percentile(C_mat, 75, axis=0)
        C_p95 = np.percentile(C_mat, 95, axis=0)
        C_mean = np.mean(C_mat, axis=0)
        Cmax_all = C_mat.max(axis=1)

        for t_p, C_p in curves:
            ax_pop.plot(t_p, C_p, color=COLORS['plasma'], alpha=0.04, lw=0.7)
        ax_pop.fill_between(t_ref, C_p5,  C_p95, alpha=0.15, color=COLORS['plasma'])
        ax_pop.fill_between(t_ref, C_p25, C_p75, alpha=0.25, color=COLORS['plasma'])
        ax_pop.plot(t_ref, C_mean, color='white', lw=2, label='Mean')
        ax_pop.axhline(MEC, color='#f97b6b', lw=1.1, ls='--', alpha=0.75)
        ax_pop.axhline(MTC, color='#f7c97e', lw=1.1, ls='--', alpha=0.75)
        ax_pop.set_xlabel('Time (hr)')
        ax_pop.set_ylabel(f'Concentration ({p["unit"]})')
        ax_pop.set_title('Population Variability (n=150)', fontsize=9)
        ax_pop.set_ylim(bottom=0)
        ax_pop.legend(fontsize=8)

        # Cmax histogram
        counts, bins, patches = ax_hist.hist(
            Cmax_all, bins=25, color=COLORS['plasma'],
            alpha=0.75, edgecolor='#1a1a2e', lw=0.5)
        for patch, left in zip(patches, bins[:-1]):
            if left > MTC:
                patch.set_facecolor('#f7c97e')
            elif left < MEC:
                patch.set_facecolor('#f97b6b')
        ax_hist.axvline(MEC, color='#f97b6b', lw=1.3, ls='--')
        ax_hist.axvline(MTC, color='#f7c97e', lw=1.3, ls='--')
        ax_hist.axvline(np.median(Cmax_all), color='white', lw=1.5, ls=':',
                        label=f'Median = {np.median(Cmax_all):.2f}')
        pct_toxic = np.mean(Cmax_all > MTC) * 100
        pct_sub   = np.mean(Cmax_all < MEC) * 100
        ax_hist.set_xlabel(f'Peak Cmax ({p["unit"]})')
        ax_hist.set_ylabel('Patients')
        ax_hist.set_title(f'Cmax Distribution\nToxic: {pct_toxic:.0f}%  Sub-therapeutic: {pct_sub:.0f}%',
                          fontsize=9)
        ax_hist.legend(fontsize=8)
    else:
        for ax in [ax_pop, ax_hist]:
            ax.text(0.5, 0.5,
                    'Population sim\nnot available\nfor IV route' if is_iv else 'Enable population\nsim above',
                    transform=ax.transAxes, ha='center', va='center',
                    color='#666688', fontsize=11)
            ax.set_xticks([])
            ax.set_yticks([])

    # ── Title & notes ─────────────────────────────────────────────────────────
    fig.text(0.5, 0.965,
             f'Drug Dashboard — {drug_name}',
             ha='center', fontsize=16, fontweight='bold', color='#e0e0ff')
    fig.text(0.5, 0.935,
             p['notes'],
             ha='center', fontsize=8.5, color='#9999bb', style='italic',
             wrap=True)

    plt.show()

# ── Widget layout ──────────────────────────────────────────────────────────────
style  = {'description_width': '160px'}
layout = widgets.Layout(width='520px')

w_drug = widgets.Dropdown(
    options=list(DRUGS.keys()),
    value='Ibuprofen',
    description='Drug',
    style=style, layout=layout
)
w_doses = widgets.IntSlider(
    value=6, min=2, max=20, step=1,
    description='Number of doses',
    style=style, layout=layout
)
w_dose_override = widgets.FloatSlider(
    value=0, min=0, max=2000, step=10,
    description='Custom dose (mg, 0=default)',
    style=style, layout=layout
)
w_population = widgets.Checkbox(
    value=True,
    description='Show population simulation',
    style=style
)

out = widgets.interactive_output(
    drug_dashboard,
    {
        'drug_name':      w_drug,
        'n_doses':        w_doses,
        'dose_override':  w_dose_override,
        'show_population': w_population,
    }
)

display(widgets.VBox([
    widgets.HTML('<h3 style="color:#e0e0ff; font-family:monospace;">Drug PK Dashboard</h3>'),
    w_drug,
    w_doses,
    w_dose_override,
    w_population,
    out
]))